In [ ]:
 -- Q1. What is a Common Table Expression (CTE), and how does it improve SQL query readability?

A Common Table Expression (CTE) is a temporary named result set that exists only during the execution of a query. It is defined using the WITH clause.

Benefits:
Improves query readability.
Breaks complex queries into smaller logical parts.
Can be referenced multiple times within the same query.
Useful for recursive queries.
Example
WITH HighSalaryEmployees AS (
    SELECT EmpID, Name, Salary
    FROM Employees
    WHERE Salary > 60000
)
SELECT *
FROM HighSalaryEmployees;
-- Q2. Why are some views updatable while others are read-only? Explain with an example.
Updatable Views

A view is updatable when it:

Is based on a single table.
Does not contain aggregate functions.
Does not contain GROUP BY, DISTINCT, UNION, or JOINs.
CREATE VIEW vw_EmployeeBasic AS
SELECT EmpID, Name, Salary
FROM Employees;

Update through the view:

UPDATE vw_EmployeeBasic
SET Salary = 70000
WHERE EmpID = 101;
Read-Only Views
CREATE VIEW vw_DepartmentSummary AS
SELECT DepartmentID,
       COUNT(*) AS TotalEmployees
FROM Employees
GROUP BY DepartmentID;

This view is read-only because it contains an aggregate function and GROUP BY.

-- Q3. What advantages do stored procedures offer compared to writing raw SQL queries repeatedly?
Advantages
Code Reusability
Write once, execute many times.
Better Performance
Execution plans are cached.
Security
Users can execute procedures without direct table access.
Reduced Network Traffic
One procedure call can execute multiple SQL statements.
Easy Maintenance
Logic changes in one place.
Example
CALL GetProductsByCategory('Electronics');

instead of repeatedly writing:

SELECT *
FROM Products
WHERE Category = 'Electronics';
-- Q4. What is the purpose of triggers in a database? Mention one use case where a trigger is essential.

A trigger is a database object that automatically executes when an INSERT, UPDATE, or DELETE event occurs on a table.

Common Uses
Audit logging
Data validation
Maintaining history tables
Synchronizing related tables
Essential Use Case

When a product is deleted, automatically save its details in an archive table.

AFTER DELETE Trigger



-- Q5. Explain the need for data modelling and normalization when designing a database.

CREATE TABLE Products (
    ProductID INT PRIMARY KEY,
    ProductName VARCHAR(100),
    Category VARCHAR(50),
    Price DECIMAL(10,2),
    Quantity INT
);
-- Q6. Write a CTE to calculate the total revenue for each product and return only products where revenue > 3000.


WITH ProductRevenue AS (
    SELECT
        ProductID,
        ProductName,
        Price,
        Quantity,
        (Price * Quantity) AS Revenue
    FROM Products
)
SELECT *
FROM ProductRevenue
WHERE Revenue > 3000;
-- Q7. Create a view named vw_CategorySummary


CREATE VIEW vw_CategorySummary AS
SELECT
    Category,
    COUNT(*) AS TotalProducts,
    AVG(Price) AS AveragePrice
FROM Products
GROUP BY Category;

-- View data:

SELECT *
FROM vw_CategorySummary;
-- Q8. Create an updatable view containing ProductID, ProductName, and Price. Then update the price of ProductID = 1 using the view.
-- Create View
CREATE VIEW vw_ProductDetails AS
SELECT
    ProductID,
    ProductName,
    Price
FROM Products;
-- Update Through View
UPDATE vw_ProductDetails
SET Price = 1200
WHERE ProductID = 1;
-- Verify
SELECT *
FROM Products
WHERE ProductID = 1;
-- Q9. Create a stored procedure that accepts a category name and returns all products belonging to that category.
MySQL
DELIMITER $$

CREATE PROCEDURE GetProductsByCategory(
    IN p_Category VARCHAR(50)
)
BEGIN
    SELECT *
    FROM Products
    WHERE Category = p_Category;
END $$

DELIMITER ;
-- Execute
CALL GetProductsByCategory('Electronics');
-- Q10. Create an AFTER DELETE trigger on the Products table that archives deleted product rows into ProductArchive.
-- Create Archive Table
CREATE TABLE ProductArchive (
    ProductID INT,
    ProductName VARCHAR(100),
    Category VARCHAR(50),
    Price DECIMAL(10,2),
    DeletedAt DATETIME
);
Create Trigger
DELIMITER $$

CREATE TRIGGER trg_ProductArchive
AFTER DELETE
ON Products
FOR EACH ROW
BEGIN
    INSERT INTO ProductArchive (
        ProductID,
        ProductName,
        Category,
        Price,
        DeletedAt
    )
    VALUES (
        OLD.ProductID,
        OLD.ProductName,
        OLD.Category,
        OLD.Price,
        NOW()
    );
END $$

DELIMITER ;
-- Test
DELETE FROM Products
WHERE ProductID = 1;
-- Verify Archive
SELECT *
FROM ProductArchive;